# 13 — Är glasögon obalanserade mellan mustache/clean i dataset_v3?
Använder CelebAs egen `Eyeglasses`-etikett (ingen egen detektor behövs) för att räkna andel glasögonbärare i varje klass i den binära mustasch-detektorns träningsdata.

In [ ]:
import os
import pandas as pd

DATASET_DIR = 'data/dataset_v3'
ATTR_PATH = 'data/list_attr_celeba.csv'

attrs = pd.read_csv(ATTR_PATH)
attrs = attrs.set_index('image_id')
print(f'{len(attrs)} bilder i attributfilen.')
print('Eyeglasses-värden:', attrs["Eyeglasses"].unique())

## Räkna glasögonandel per klass

In [ ]:
def collect_files(folder):
    paths = []
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                paths.append(os.path.join(root, f))
    return paths

results = {}

for cls in os.listdir(DATASET_DIR):
    cls_dir = os.path.join(DATASET_DIR, cls)
    if not os.path.isdir(cls_dir):
        continue

    files = collect_files(cls_dir)
    total = 0
    with_glasses = 0
    not_found = 0

    for path in files:
        fname = os.path.basename(path)
        if fname not in attrs.index:
            not_found += 1
            continue
        total += 1
        if attrs.loc[fname, 'Eyeglasses'] == 1:
            with_glasses += 1

    results[cls] = {
        'totalt_med_attribut': total,
        'med_glasogon': with_glasses,
        'andel_glasogon_procent': round(with_glasses / total * 100, 2) if total > 0 else None,
        'ej_hittade_i_attributfil': not_found,
    }

pd.DataFrame(results).T

## Tolkning
Om andelen glasögonbärare skiljer sig markant mellan `mustache` och `clean` (t.ex. dubbelt så vanligt i en av klasserna), är det en genuin databalans-bias värd att fixa — antingen genom att lägga till fler glasögonbärare i underrepresenterade klassen, eller ta bort några från den överrepresenterade.